# **Implementación del Algoritmo de Shor para resolver el  DLP**

## Oráculo de Beauregard

### Rodrigo Hernández Sacristán

#### Máster en Computación Cuántica, Universidad Internacional de la Rioja (UNIR)

In [1]:
## LIBRERÍAS NECESARIAS 

import numpy as np
import math
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate 
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit.visualization import plot_histogram
from IPython.display import display
from qiskit.quantum_info import Operator
from qiskit.circuit.library import QFT
import warnings
import random

In [2]:
"""
Oráculo cuántico para el algoritmo de Shor (logaritmo discreto) que
implementa la EXPONENCIACIÓN MODULAR mediante puertas cuánticas
(sumadores de Draper en espacio de Fourier + esquema de Beauregard),
en sustitución de la versión basada en matrices de permutación 2^l x 2^l.

    |a>|b>|0...0>  -->  |a>|b>| g^a * h^(-b) mod p >

El registro de valor (reg_C) se multiplica EN SITIO, primero por las
potencias de g (controladas por reg_A) y después por las potencias de
h^(-1) (controladas por reg_B). Las ancillas empiezan y terminan en |0>.

Coste de qubits del registro inferior (para módulo p):
    size      = ceil(log2(p))          # qubits para el valor  
    n_ancilla = size + 2               # ancillas auxiliares    
Por ejemplo, para p = 23:  size = 5,  n_ancilla = 7.

==================================================================
"""


# ------------------------------------------------------------------
# QFT sin swaps 
# ------------------------------------------------------------------
def _qft_ns_brg(n):
    qc = QuantumCircuit(n, name="QFT")
    for j in range(n - 1, -1, -1):
        qc.h(j)
        for k in range(j - 1, -1, -1):
            qc.cp(math.pi / 2 ** (j - k), k, j)
    return qc


def _angle_brg(k, b):
    """Ángulo de rotación de Draper: b*pi / 2^k."""
    return b * np.pi / (2 ** k)


# ------------------------------------------------------------------
# Sumadores en espacio de Fourier 
# ------------------------------------------------------------------
def _phi_add_brg(n, b, factor):
    """|phi(x)> -> |phi(x + factor*b)>  (sin control)."""
    qc = QuantumCircuit(n, name=f"add({b})")
    for k in range(n):
        qc.p(factor * _angle_brg(k, b), k)
    return qc


def _phi_add_c_brg(n, b, factor):
    """Sumador con 1 control."""
    ctrl = QuantumRegister(1, "c")
    reg = QuantumRegister(n, "r")
    qc = QuantumCircuit(ctrl, reg, name=f"cadd({b})")
    for k in range(n):
        qc.cp(factor * _angle_brg(k, b), ctrl[0], reg[k])
    return qc


def _phi_add_cc_brg(n, b, factor):
    """Sumador con 2 controles."""
    ctrl = QuantumRegister(2, "c")
    reg = QuantumRegister(n, "r")
    qc = QuantumCircuit(ctrl, reg, name=f"ccadd({b})")
    for k in range(n):
        qc.mcp(factor * _angle_brg(k, b), [ctrl[0], ctrl[1]], reg[k])
    return qc


# ------------------------------------------------------------------
# Sumador modular doble-controlado  (Beauregard)
#   |phi(a)>|0>_aux -> |phi((a+y) mod N)>|0>_aux
#   El registro 'reg' entra y sale en espacio de Fourier.
# ------------------------------------------------------------------
def _phi_add_mod_N_cc_brg(n, y, N):
    ctrl = QuantumRegister(2, "ctrl")
    reg = QuantumRegister(n, "a")
    aux = QuantumRegister(1, "aux")
    qc = QuantumCircuit(ctrl, reg, aux, name=f"ccADD({y})MOD({N})")

    iqft = _qft_ns_brg(n).inverse()
    qft = _qft_ns_brg(n)

    qc.append(_phi_add_cc_brg(n, y, 1).to_gate(), list(ctrl) + list(reg))
    qc.append(_phi_add_brg(n, N, -1).to_gate(), reg)
    qc.append(iqft.to_gate(), reg)
    qc.cx(reg[n - 1], aux[0])
    qc.append(qft.to_gate(), reg)
    qc.append(_phi_add_c_brg(n, N, 1).to_gate(), [aux[0]] + list(reg))
    qc.append(_phi_add_cc_brg(n, y, -1).to_gate(), list(ctrl) + list(reg))
    qc.append(iqft.to_gate(), reg)
    qc.x(reg[n - 1])
    qc.cx(reg[n - 1], aux[0])
    qc.x(reg[n - 1])
    qc.append(qft.to_gate(), reg)
    qc.append(_phi_add_cc_brg(n, y, 1).to_gate(), list(ctrl) + list(reg))
    return qc


# ------------------------------------------------------------------
# Multiplicación modular controlada (en sitio)
#   |c>|x>|0>|0> -> |c>| (x * a mod N) si c=1, si no x >|0>|0>
# ------------------------------------------------------------------
def _mult_mod_N_partial_c_brg(y, N, size_x, size_b):
    """Multiplicación-suma fuera de sitio: |c>|x>|b>|anc> -> |c>|x>|b + y*x mod N>|anc>."""
    ctrl = QuantumRegister(1, "c")
    rx = QuantumRegister(size_x, "x")
    rb = QuantumRegister(size_b, "b")
    anc = QuantumRegister(1, "anc")
    qc = QuantumCircuit(ctrl, rx, rb, anc, name=f"CMUL0({y})")

    qc.append(_qft_ns_brg(size_b).to_gate(), rb)
    for k in range(size_x):
        add = _phi_add_mod_N_cc_brg(size_b, (2 ** k * y) % N, N)
        qc.append(add.to_gate(), [ctrl[0], rx[k]] + list(rb) + [anc[0]])
    qc.append(_qft_ns_brg(size_b).inverse().to_gate(), rb)
    return qc


def _mult_mod_N_c_brg(a, N):
    """Multiplicación modular controlada EN SITIO: |c>|x>|0>|0> -> |c>| a*x mod N >|0>|0>.
    Requiere gcd(a, N) = 1 (siempre cierto aquí porque N = p es primo)."""
    size = math.ceil(math.log2(N))
    size_x = size
    size_b = size + 1

    ctrl = QuantumRegister(1, "c")
    rx = QuantumRegister(size_x, "x")
    rb = QuantumRegister(size_b, "b")
    anc = QuantumRegister(1, "anc")
    qc = QuantumCircuit(ctrl, rx, rb, anc, name=f"CMUL({a})MOD({N})")

    qc.append(_mult_mod_N_partial_c_brg(a, N, size_x, size_b).to_gate(), qc.qubits)
    for i in range(size_x):
        qc.cswap(ctrl[0], rx[i], rb[i])
    inv_a = pow(a, -1, N)
    qc.append(_mult_mod_N_partial_c_brg(inv_a, N, size_x, size_b).inverse().to_gate(), qc.qubits)
    return qc


# ------------------------------------------------------------------
# Exponenciación modular:  |x>|input> -> |x>| (valor * a^x) mod N >
#   El registro 'input' tiene 2*ceil(log2 N)+2 qubits:
#     [ valor (size) | b (size+1) | anc (1) ]
#   El valor vive en los primeros 'size' qubits de 'input'.
# ------------------------------------------------------------------
def _mod_exp_brg(n, a, N):
    size = math.ceil(math.log2(N))
    size_input = 2 * size + 2
    xr = QuantumRegister(n, "x")
    inp = QuantumRegister(size_input, "input")
    qc = QuantumCircuit(xr, inp, name=f"{a}^x mod {N}")
    for i in range(n):
        cte = pow(a, 2 ** i, N)   # constante a^(2^i) mod N para este bit del exponente
        # multiplicar por 1 es la identidad, así que nos saltamos por completo esa multiplicación controlada.
        if cte == 1:
            continue
        g = _mult_mod_N_c_brg(cte, N)
        qc.append(g.to_gate(), [xr[i]] + list(inp))
    return qc


def num_ancillas_beauregard(p):
    """Número de qubits ancilla que necesita el oráculo para el módulo p."""
    return math.ceil(math.log2(p)) + 2


def aplicar_oraculo_beauregard(g, h, p, qc, reg_A, reg_B, reg_C, reg_anc):
    """
    Implementa f(a,b) = g^a * h^(-b) mod p mediante puertas cuánticas.

        |a>|b>|1>|0...0>  -->  |a>|b>| g^a * h^(-b) mod p >|0...0>

    Parámetros
    ----------
    reg_A, reg_B : registros de exponente (l qubits cada uno).
    reg_C        : registro de VALOR (l qubits). Debe cumplir l >= ceil(log2 p).
    reg_anc      : registro ancilla NUEVO. Tamaño = num_ancillas_beauregard(p)
                   (para p=23 -> 7 qubits). Entra y sale en |0>.
    """
    l = len(reg_A)
    size = math.ceil(math.log2(p))
    assert len(reg_C) >= size, f"reg_C necesita >= {size} qubits para p={p}"
    assert len(reg_anc) == num_ancillas_beauregard(p), \
        f"reg_anc debe tener {num_ancillas_beauregard(p)} qubits para p={p}, tiene {len(reg_anc)}"

    # El 'input' que espera _mod_exp es [valor(size) | b(size+1) | anc(1)].
    # Aquí: valor = reg_C, y el resto (size+1 + 1 = size+2) son las ancillas.
    bottom = list(reg_C) + list(reg_anc)

    # Inicializamos el registro de valor a |1> (empezamos a multiplicar desde 1)
    qc.x(reg_C[0])

    h_inv = pow(h, -1, p)
    print(f"[Cuántico] Exponenciación por puertas: base g={g} (reg_A), base h^-1={h_inv} (reg_B)")

    # 1) Multiplica en sitio por g^a  (controlado por reg_A)
    qc.append(_mod_exp_brg(l, g, p).to_gate(label=f"g^a mod {p}"), list(reg_A) + bottom)

    # 2) Multiplica en sitio por (h^-1)^b = h^-b  (controlado por reg_B)
    qc.append(_mod_exp_brg(l, h_inv, p).to_gate(label=f"h^-b mod {p}"), list(reg_B) + bottom)
